In [1]:

# Importer les bibliothèques nécessaires au Matching Agent.
# Vérifier que notre environnement Python est prêt pour LangGraph.

import sys
import pydantic
import langgraph

from typing import TypedDict, Optional
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

print("=== Matching Agent : Initialisation ===")
print("Python :", sys.version.split()[0])
print("Pydantic :", pydantic.__version__)
print("LangGraph : import réussi")
print("Environnement prêt !")


=== Matching Agent : Initialisation ===
Python : 3.11.17
Pydantic : 2.14.0
LangGraph : import réussi
Environnement prêt !


In [2]:

# Définir les modèles d'entrée utilisés par le Matching Agent.
# Conserver les champs nécessaires à la comparaison candidat-offres.

from pydantic import BaseModel, Field, HttpUrl
from typing import Optional


class CandidateProfile(BaseModel):
    full_name: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None
    location: Optional[str] = None

    skills: list[str] = Field(default_factory=list)
    education: list = Field(default_factory=list)
    experience: list = Field(default_factory=list)
    languages: list[str] = Field(default_factory=list)

    target_roles: list[str] = Field(default_factory=list)
    employment_types: list[str] = Field(default_factory=list)


class JobOffer(BaseModel):
    source: str
    source_id: str

    title: str
    company: str
    location: Optional[str] = None
    description: Optional[str] = None

    tags: list[str] = Field(default_factory=list)
    job_type: Optional[str] = None
    category: Optional[str] = None
    salary: Optional[str] = None

    publication_date: Optional[str] = None
    url: HttpUrl


print("Modèle CandidateProfile prêt !")
print("Modèle JobOffer prêt !")


Modèle CandidateProfile prêt !
Modèle JobOffer prêt !


In [3]:

# Créer un profil candidat fictif pour tester le Matching Agent.
# Préparer trois offres de pertinence différente pour vérifier le filtrage.

test_candidate = CandidateProfile(
    full_name="Test Candidate",
    location="Tunisia",
    skills=["Python", "FastAPI", "Docker", "SQL", "Machine Learning"],
    languages=["French", "English"],
    target_roles=["Python Developer", "AI Engineer"],
    employment_types=["full_time"]
)

test_jobs = [
    JobOffer(
        source="remotive",
        source_id="test-001",
        title="Python Backend Developer",
        company="Tech Solutions",
        location="Worldwide",
        description="Develop backend APIs using Python, FastAPI, SQL and Docker.",
        tags=["Python", "FastAPI", "Docker", "SQL"],
        job_type="full_time",
        url="https://example.com/jobs/001"
    ),
    JobOffer(
        source="arbeitnow",
        source_id="test-002",
        title="AI Engineer",
        company="AI Innovations",
        location="Remote",
        description="Build machine learning models using Python and ML frameworks.",
        tags=["Python", "Machine Learning", "AI"],
        job_type="full_time",
        url="https://example.com/jobs/002"
    ),
    JobOffer(
        source="remotive",
        source_id="test-003",
        title="Content Writer",
        company="Media Group",
        location="Worldwide",
        description="Write articles and blog posts for online publications.",
        tags=["Writing", "Content", "SEO"],
        job_type="freelance",
        url="https://example.com/jobs/003"
    )
]

print("=== Données de test ===")
print("Candidat :", test_candidate.full_name)
print("Compétences :", test_candidate.skills)
print("Métiers ciblés :", test_candidate.target_roles)
print("Nombre d'offres :", len(test_jobs))

for job in test_jobs:
    print(f"- {job.title} | {job.company}")


=== Données de test ===
Candidat : Test Candidate
Compétences : ['Python', 'FastAPI', 'Docker', 'SQL', 'Machine Learning']
Métiers ciblés : ['Python Developer', 'AI Engineer']
Nombre d'offres : 3
- Python Backend Developer | Tech Solutions
- AI Engineer | AI Innovations
- Content Writer | Media Group


In [4]:

# Définir la structure du résultat produit par le LLM Matching Agent.
# Valider le score, la pertinence et les explications avec Pydantic.

from pydantic import BaseModel, Field

class MatchResult(BaseModel):

    job_id: str = Field(
        description="Identifiant unique de l'offre analysée"
    )

    match_score: int = Field(
        ge=0,
        le=100,
        description="Score de compatibilité entre 0 et 100"
    )

    is_relevant: bool = Field(
        description="Indique si l'offre correspond au candidat"
    )

    matching_skills: list[str] = Field(
        default_factory=list,
        description="Compétences du candidat correspondant au poste"
    )

    missing_skills: list[str] = Field(
        default_factory=list,
        description="Compétences exigées mais absentes du profil"
    )

    explanation: str = Field(
        description="Explication concise du score attribué"
    )


print("=== Modèle MatchResult ===")

for field_name in MatchResult.model_fields:
    print("-", field_name)

print("\nModèle créé avec succès !")


=== Modèle MatchResult ===
- job_id
- match_score
- is_relevant
- matching_skills
- missing_skills
- explanation

Modèle créé avec succès !


In [6]:

# Charger les trois clés OpenRouter depuis le fichier .env.
# Initialiser le Matching Agent avec la première clé disponible.

import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Charger le fichier .env
for env_path in [Path.cwd() / ".env", Path.cwd().parent / ".env"]:
    if env_path.is_file():
        load_dotenv(env_path)
        break

# Récupérer les trois clés
api_keys = [
    os.getenv("OPENROUTER_API_KEY_1"),
    os.getenv("OPENROUTER_API_KEY_2"),
    os.getenv("OPENROUTER_API_KEY_3")
]

api_keys = [key for key in api_keys if key]

if not api_keys:
    raise ValueError("Aucune clé OpenRouter trouvée dans .env")

print("Nombre de clés disponibles :", len(api_keys))

# Utiliser la première clé
llm = ChatOpenAI(
    model=os.getenv("OPENROUTER_MODEL"),
    api_key=api_keys[0],
    base_url=os.getenv(
        "OPENROUTER_BASE_URL",
        "https://openrouter.ai/api/v1"
    ),
    temperature=0
)

matching_llm = llm.with_structured_output(
    MatchResult,
    method="function_calling"
)

print("LLM Matching Agent configuré avec succès !")


Nombre de clés disponibles : 3
LLM Matching Agent configuré avec succès !


In [7]:

# Construire le prompt qui guide l'analyse de compatibilité par le LLM.
# Comparer le profil du candidat et l'offre selon des critères explicites.

from langchain_core.prompts import ChatPromptTemplate

matching_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Tu es un expert en recrutement technique et en matching CV-offres d'emploi.

Ta mission est d'évaluer la compatibilité entre un candidat et une offre.

Analyse les critères suivants :
1. Correspondance entre le poste et les métiers ciblés.
2. Adéquation des compétences techniques.
3. Correspondance de l'expérience et de la formation.
4. Compatibilité de la localisation et du type de contrat.

Attribue un score global de 0 à 100 :
- 0 à 39 : faible compatibilité
- 40 à 69 : compatibilité moyenne
- 70 à 100 : bonne compatibilité

Règles :
- Utilise uniquement les informations fournies.
- N'invente aucune compétence ou expérience.
- Une information absente n'est pas nécessairement une incompatibilité.
- Identifie les compétences correspondantes et manquantes.
- Considère une offre pertinente si le score est supérieur ou égal à 70.
- Retourne le source_id de l'offre dans job_id.
- Explique brièvement la justification du score.
- Réponds en français.
"""
    ),
    (
        "human",
        """
PROFIL DU CANDIDAT :
{candidate_profile}

OFFRE D'EMPLOI :
{job_offer}

Évalue la compatibilité de cette offre avec le candidat.
"""
    )
])

print("Prompt du Matching Agent créé avec succès !")
print("Variables attendues :", matching_prompt.input_variables)


Prompt du Matching Agent créé avec succès !
Variables attendues : ['candidate_profile', 'job_offer']


In [8]:

# Construire la chaîne LangChain pour analyser la compatibilité candidat-offre.
# Tester le LLM sur une première offre et afficher son résultat structuré.

from langchain_core.output_parsers import PydanticOutputParser

# Assembler le prompt et le LLM
matching_chain = matching_prompt | matching_llm

# Sélectionner la première offre
job = test_jobs[0]

# Appeler le LLM
result = matching_chain.invoke({
    "candidate_profile": test_candidate.model_dump_json(indent=2),
    "job_offer": job.model_dump_json(indent=2)
})

# Afficher le résultat
print("=== Résultat du Matching Agent ===")
print("Offre :", job.title)
print("Entreprise :", job.company)
print("Score :", result.match_score, "/100")
print("Pertinente :", result.is_relevant)
print("Compétences correspondantes :", result.matching_skills)
print("Compétences manquantes :", result.missing_skills)
print("Explication :", result.explanation)

print("\nType de résultat :", type(result).__name__)


=== Résultat du Matching Agent ===
Offre : Python Backend Developer
Entreprise : Tech Solutions
Score : 95 /100
Pertinente : True
Compétences correspondantes : ['Python', 'FastAPI', 'Docker', 'SQL']
Compétences manquantes : []
Explication : Excellente correspondance : le poste de Python Backend Developer correspond au rôle cible 'Python Developer'. Toutes les compétences techniques requises (Python, FastAPI, Docker, SQL) sont présentes. La localisation 'Worldwide' inclut la Tunisie et le type de contrat 'full_time' correspond aux préférences. Le candidat possède également une compétence bonus (Machine Learning) non requise mais valorisante.

Type de résultat : MatchResult


In [9]:

# Analyser chaque offre avec le LLM et le profil du candidat.
# Classer les résultats par score de compatibilité décroissant.

matching_results = []

for index, job in enumerate(test_jobs, start=1):
    print(f"Analyse {index}/{len(test_jobs)} : {job.title}")

    try:
        result = matching_chain.invoke({
            "candidate_profile": test_candidate.model_dump_json(indent=2),
            "job_offer": job.model_dump_json(indent=2)
        })

        matching_results.append(result)

    except Exception as e:
        print(f"Erreur pour {job.title} : {e}")

# Classer les résultats
matching_results.sort(
    key=lambda result: result.match_score,
    reverse=True
)

# Associer les résultats aux offres
jobs_by_id = {
    job.source_id: job
    for job in test_jobs
}

print("\n=== CLASSEMENT DES OFFRES ===")

for rank, result in enumerate(matching_results, start=1):
    job = jobs_by_id.get(result.job_id)

    if job is None:
        print(f"Résultat avec identifiant inconnu : {result.job_id}")
        continue

    print(f"\n{rank}. {job.title}")
    print(f"   Entreprise : {job.company}")
    print(f"   Score : {result.match_score}/100")
    print(f"   Pertinente : {result.is_relevant}")
    print(f"   Explication : {result.explanation}")


Analyse 1/3 : Python Backend Developer
Analyse 2/3 : AI Engineer
Analyse 3/3 : Content Writer

=== CLASSEMENT DES OFFRES ===

1. Python Backend Developer
   Entreprise : Tech Solutions
   Score : 95/100
   Pertinente : True
   Explication : Le candidat possède toutes les compétences techniques requises (Python, FastAPI, Docker, SQL). Le poste correspond à son rôle cible 'Python Developer'. La localisation 'Worldwide' est compatible avec la Tunisie (télétravail) et le type de contrat 'full_time' correspond. L'expérience en Machine Learning est un atout supplémentaire.

2. AI Engineer
   Entreprise : AI Innovations
   Score : 85/100
   Pertinente : True
   Explication : Le candidat possède une excellente adéquation avec l'offre AI Engineer. Il maîtrise tous les compétences clés requises : Python, Machine Learning et AI (via ses compétences en ML). Les compétences supplémentaires comme FastAPI, Docker et SQL renforcent son profil pour ce rôle. La localisation 'Remote' est compatible avec 

In [10]:

# Définir les données partagées entre les nœuds du Matching Agent.
# Stocker le candidat, les offres, les résultats et le classement.

from typing import TypedDict
from pydantic import Field

class MatchingState(TypedDict):
    candidate_profile: CandidateProfile
    job_offers: list[JobOffer]
    matching_results: list[MatchResult]
    ranked_jobs: list[JobOffer]
    errors: list[str]


print("=== MatchingState ===")

for field_name, field_type in MatchingState.__annotations__.items():
    print(f"{field_name} : {field_type}")

print("\nMatchingState créé avec succès !")



=== MatchingState ===
candidate_profile : <class '__main__.CandidateProfile'>
job_offers : list[__main__.JobOffer]
matching_results : list[__main__.MatchResult]
ranked_jobs : list[__main__.JobOffer]
errors : list[str]

MatchingState créé avec succès !


In [13]:

# Analyser les offres avec le LLM et réessayer en cas de réponse vide.
# Enregistrer uniquement les résultats valides et conserver les erreurs.

import time

def match_jobs(state: MatchingState) -> dict:

    candidate = state["candidate_profile"]
    jobs = state["job_offers"]

    results = []
    errors = list(state.get("errors", []))

    print(f"=== Matching Agent : {len(jobs)} offres à analyser ===")

    for index, job in enumerate(jobs, start=1):

        print(f"\n[{index}/{len(jobs)}] {job.title}")

        for attempt in range(1, 4):

            try:
                result = matching_chain.invoke({
                    "candidate_profile": candidate.model_dump_json(),
                    "job_offer": job.model_dump_json()
                })

                if result is None:
                    raise ValueError(
                        "Le LLM a retourné une réponse vide."
                    )

                if not isinstance(result, MatchResult):
                    raise TypeError(
                        "La réponse ne respecte pas MatchResult."
                    )

                result.job_id = job.source_id

                results.append(result)

                print(f"Score : {result.match_score}/100")
                print(f"Pertinente : {result.is_relevant}")

                break

            except Exception as e:

                print(f"Tentative {attempt}/3 échouée : {e}")

                if attempt == 3:
                    errors.append(
                        f"{job.title} : {str(e)}"
                    )
                else:
                    time.sleep(2)

    return {
        "matching_results": results,
        "errors": errors
    }


print("Nœud match_jobs corrigé avec succès !")


Nœud match_jobs corrigé avec succès !


In [14]:

# Préparer un état initial avec le candidat et les offres de test.
# Exécuter le nœud match_jobs et vérifier les résultats du LLM.

initial_state = {
    "candidate_profile": test_candidate,
    "job_offers": test_jobs,
    "matching_results": [],
    "ranked_jobs": [],
    "errors": []
}

# Exécuter le nœud
matching_output = match_jobs(initial_state)

# Afficher le bilan
print("\n=== BILAN DU MATCHING ===")
print("Offres analysées :", len(test_jobs))
print("Résultats obtenus :", len(matching_output["matching_results"]))
print("Erreurs :", len(matching_output["errors"]))

print("\n=== SCORES ===")
for result in matching_output["matching_results"]:
    print(
        f"{result.job_id} | "
        f"Score : {result.match_score}/100 | "
        f"Pertinente : {result.is_relevant}"
    )


=== Matching Agent : 3 offres à analyser ===

[1/3] Python Backend Developer
Score : 95/100
Pertinente : True

[2/3] AI Engineer
Tentative 1/3 échouée : Le LLM a retourné une réponse vide.
Score : 80/100
Pertinente : True

[3/3] Content Writer
Tentative 1/3 échouée : Le LLM a retourné une réponse vide.
Score : 35/100
Pertinente : False

=== BILAN DU MATCHING ===
Offres analysées : 3
Résultats obtenus : 3
Erreurs : 0

=== SCORES ===
test-001 | Score : 95/100 | Pertinente : True
test-002 | Score : 80/100 | Pertinente : True
test-003 | Score : 35/100 | Pertinente : False


In [15]:

# Classer les offres selon les scores calculés par le LLM.
# Conserver uniquement les offres pertinentes dans le classement final.

def rank_jobs(state: MatchingState) -> dict:

    jobs = state["job_offers"]
    results = state["matching_results"]

    # Associer chaque résultat à son offre
    jobs_by_id = {
        job.source_id: job
        for job in jobs
    }

    # Garder uniquement les résultats pertinents
    relevant_results = [
        result
        for result in results
        if result.is_relevant and result.match_score >= 70
    ]

    # Trier par score décroissant
    relevant_results.sort(
        key=lambda result: result.match_score,
        reverse=True
    )

    # Construire le classement des offres
    ranked_jobs = [
        jobs_by_id[result.job_id]
        for result in relevant_results
        if result.job_id in jobs_by_id
    ]

    print("=== CLASSEMENT FINAL ===")
    print("Offres initiales :", len(jobs))
    print("Offres pertinentes :", len(ranked_jobs))

    for rank, result in enumerate(relevant_results, start=1):
        job = jobs_by_id.get(result.job_id)

        if job is not None:
            print(
                f"{rank}. {job.title} "
                f"| Score : {result.match_score}/100"
            )

    return {
        "ranked_jobs": ranked_jobs
    }


print("\nNœud rank_jobs créé avec succès !")



Nœud rank_jobs créé avec succès !


In [16]:

# Construire le graphe LangGraph du Matching Agent.
# Connecter l'analyse LLM au classement des offres pertinentes.

from langgraph.graph import StateGraph, START, END

# Initialiser le graphe avec notre State
matching_builder = StateGraph(MatchingState)

# Ajouter les deux nœuds
matching_builder.add_node("match_jobs", match_jobs)
matching_builder.add_node("rank_jobs", rank_jobs)

# Définir l'ordre d'exécution
matching_builder.add_edge(START, "match_jobs")
matching_builder.add_edge("match_jobs", "rank_jobs")
matching_builder.add_edge("rank_jobs", END)

# Compiler le graphe
matching_graph = matching_builder.compile()

print("=== Matching Agent ===")
print("Graphe LangGraph compilé avec succès !")

# Afficher le graphe
display(matching_graph.get_graph().draw_mermaid_png())


=== Matching Agent ===
Graphe LangGraph compilé avec succès !


b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x80\x00\x00\x01M\x08\x02\x00\x00\x00G]\xf0B\x00\x00\x10\x00IDATx\x9c\xec\x9d\x07\\\x93G\x1f\xc7/{\xb1\xa7 [p\x81\x82\x1b\xfbZ\x17\xe2\xde\xa3ZT\x1c\xb5\xae\xda\xd6U[W\x1duT\xab\xd6Z\xab\xd6j\xb5\xee\xad\xad\x13\xf7D\xdc\xb8A\x11\x04D6\x84\x99\x90\x90\xe4\xfdC\x14\x10\x03B\xf2<\x1c\x0f\xdeW?\xf9\xdcswy\xf2p\xbf\xbb\xff\xed{\xb8\x1a\x8d\x06\x11\xf0\xc1F\x04\xac\x10\x010C\x04\xc0\x0c\x11\x003D\x00\xcc\x10\x010\xc3E4\xf3\xf0Zz\xcc\xd3\xdc\x9cLU\xbe\x12)\x15\xea\x92A,6K\xa3\xd6\xbc\xe3`!\xf8\xa7~\xdb2f\xb3Yj\xb5\x86\xc5bk4\xef|\x91\xc3ckT\x1a\xb5\xfa\x9d\x064D\x86\xcfR\x9e\x05\xfe\xac\xe2\x1b\x96\xbc-\x9b\xc3R\xab\xde\xf8\xb3X\xac\xa2\xe68\xdc\x85\xcdc\tD,S\x0b\x9e\x8b\x97\xa4n\x13\x13D\',\x9a\xfa\x01\xe7\xf7&F=\xce\x91\xe7\xa8\xd9l\xc4\x13\xb2y|\x16\x97\xcbV)\xdf\xfdm6*\xf8\xf5\x82\x14F\xda\x14.p@\n\xbcMD\xad0E\xf2\x14\xc1\xe1\xb2T\xe0\xa3~\xffn\x08\xbd\xff\xd7\xb0!&\xabtL5*)@A\xaa\xbfu\x82\xd8l.\xfc"\xca\x93\xa9\x94Jx<$1\xe14lm\xdc\xc2\xdf

In [17]:
final_state = matching_graph.invoke(initial_state)

=== Matching Agent : 3 offres à analyser ===

[1/3] Python Backend Developer
Score : 95/100
Pertinente : True

[2/3] AI Engineer
Score : 88/100
Pertinente : True

[3/3] Content Writer
Score : 10/100
Pertinente : False
=== CLASSEMENT FINAL ===
Offres initiales : 3
Offres pertinentes : 2
1. Python Backend Developer | Score : 95/100
2. AI Engineer | Score : 88/100


In [18]:

# Vérifier les données produites par le graphe LangGraph.
# Confirmer que les offres classées correspondent aux résultats du LLM.

print("=== VALIDATION DU MATCHING AGENT ===")

print("Profil candidat :", final_state["candidate_profile"].full_name)
print("Offres reçues :", len(final_state["job_offers"]))
print("Résultats LLM :", len(final_state["matching_results"]))
print("Offres retenues :", len(final_state["ranked_jobs"]))
print("Erreurs :", len(final_state["errors"]))

scores_by_id = {
    result.job_id: result.match_score
    for result in final_state["matching_results"]
}

print("\n=== OFFRES À TRANSMETTRE À L'APPLICATION AGENT ===")

for index, job in enumerate(final_state["ranked_jobs"], start=1):
    print(f"\n{index}. {job.title}")
    print("Entreprise :", job.company)
    print("Score :", scores_by_id[job.source_id], "/100")
    print("URL :", job.url)

assert len(final_state["matching_results"]) == 3
assert len(final_state["ranked_jobs"]) == 2
assert len(final_state["errors"]) == 0

print("\nValidation réussie !")


=== VALIDATION DU MATCHING AGENT ===
Profil candidat : Test Candidate
Offres reçues : 3
Résultats LLM : 3
Offres retenues : 2
Erreurs : 0

=== OFFRES À TRANSMETTRE À L'APPLICATION AGENT ===

1. Python Backend Developer
Entreprise : Tech Solutions
Score : 95 /100
URL : https://example.com/jobs/001

2. AI Engineer
Entreprise : AI Innovations
Score : 88 /100
URL : https://example.com/jobs/002

Validation réussie !
